# Phase 2 — SFT LoRA de Qwen3-1.7B-Base

Fine-tuning supervisé (SFT) de `Qwen/Qwen3-1.7B-Base` sur le dataset anonymisé de la phase 1, avec LoRA.

- **Prérequis** : GPU, fichiers `data/anonymized/sft_mediqal_mcqu_{train,validation,test}.anonymized.jsonl` produits par `data_prep_MediQAl`.
- **Étapes** : données → modèle → LoRA → entraînement → validation intermédiaire.
- **Suivi** : MLflow sur DagsHub (`MLFLOW_BACKEND = "dagshub"`), consultable de partout et quelle que soit la machine d'entraînement. En local (`"local"`) : `mlflow ui --backend-store-uri sqlite:///mlflow.db` depuis la racine du projet.

In [16]:
# Installation (Colab ou machine GPU) : décommenter si besoin
# %pip install -q torch transformers peft trl datasets accelerate mlflow dagshub pandas

## 1. Configuration

In [17]:
import functools
import json
import math
import re
import time
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from transformers.generation.streamers import BaseStreamer
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
# Découpage d'origine de MediQAl, anonymisé en phase 1
DATA_FILES = {
    split: f"../data/anonymized/sft_mediqal_mcqu_{split}.anonymized.jsonl"
    for split in ("train", "validation", "test")
}

# Suivi MLflow : "dagshub" (serveur distant) ou "local" (fichier ../mlflow.db)
MLFLOW_BACKEND = "dagshub"
DAGSHUB_REPO_OWNER = "obiyohan"
DAGSHUB_REPO_NAME = "my-ml-mlflow"

# Petit run pour valider la pipeline avant l'entraînement complet
SMALL_RUN = False  # True pour un run rapide sur 100 exemples, False pour le run complet

SEED = 42
RUN_NAME = f"sft-lora-r16-seed{SEED}" + ("-small" if SMALL_RUN else "")
OUTPUT_DIR = Path("../outputs") / RUN_NAME  # checkpoints et adaptateur final

set_seed(SEED)
if torch.cuda.is_available():
    device, bf16 = "cuda", torch.cuda.is_bf16_supported()  # bf16 absent sur T4 : fp16 à la place
elif torch.xpu.is_available():
    device, bf16 = "xpu", torch.xpu.is_bf16_supported()  # GPU Intel Arc
else:
    device, bf16 = "cpu", False
dtype = torch.bfloat16 if bf16 else torch.float16 if device != "cpu" else torch.float32
print(f"device={device}, dtype={dtype}")

if device == "xpu":
    # GPU Intel sous Windows : deux contournements
    # 1. Pas de compilateur C pour Triton : TRL calcule la loss en PyTorch pur
    import trl.kernels.chunked_logprob as chunked_logprob
    chunked_logprob._TRITON_AVAILABLE = False
    # 2. La norme des gradients en mode « foreach » fait planter le GPU : calcul tenseur par tenseur
    torch.nn.utils.clip_grad_norm_ = functools.partial(torch.nn.utils.clip_grad.clip_grad_norm_, foreach=False)

device=cuda, dtype=torch.bfloat16


## 2. Données : train / validation / test

In [18]:
# Format prompt / completion : la loss ne porte que sur la réponse de l'assistant
def to_prompt_completion(record):
    return {"prompt": record["messages"][:-1], "completion": record["messages"][-1:]}


splits = load_dataset("json", data_files=DATA_FILES)
splits = splits.map(to_prompt_completion, remove_columns=["messages"])
for name, ds in splits.items():
    print(f"{name}: {len(ds)} exemples")

# Petit run : sous-échantillon
if SMALL_RUN:
    for name, size in {"train": 50, "validation": 15, "test": 8}.items():
        splits[name] = splits[name].select(range(size))

train: 3935 exemples
validation: 999 exemples
test: 1996 exemples


## 3. Tokenizer et modèle

In [19]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Gabarit ChatML
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

example = splits["train"][0]
print(tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False))

<|im_start|>system
Tu es un assistant médical expert. Réponds de façon précise et concise.<|im_end|>
<|im_start|>user
Monsieur <PERSON> âgé de 37 ans, forestier de son métier, doit partir travailler pendant 2 ans au Gabon dans la région de Lambaréné. Monsieur <PERSON> a l'intention de se faire accompagner de son épouse âgée de 36 ans et de son fils âgé de 5 ans. Ils logeront à quelques kilomètres de la forêt mais Mr <PERSON>, de par son métier, passera la plupart de son temps au milieu des bois. Ils consultent 4 mois avant le départ pour se renseigner au sujet des vaccinations et de la chimioprophylaxie du paludisme. Vous lui expliquez les conseils d'hygiène, les éléments de prévention essentiels et les différents modes de contamination des maladies présentes dans la région

Au sujet des vaccinations :<|im_end|>
<|im_start|>assistant
Le vaccin contre la fièvre jaune est obligatoire<|im_end|>



In [20]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"{model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2426.08it/s]


1.72 B paramètres


## 4. LoRA

In [21]:
# Seules de petites matrices de rang r sont entraînées : le modèle de base reste figé
lora_config = LoraConfig(
    r=16,                         # rang des matrices d'adaptation
    lora_alpha=32,                # facteur d'échelle (souvent 2 × r)
    lora_dropout=0.05,
    target_modules="all-linear",  # couches d'attention et MLP
    task_type="CAUSAL_LM",
)

## 5. Entraînement SFT

In [22]:
# Suivi MLflow : un run par entraînement, hyperparamètres et métriques enregistrés
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    # Jeton lu dans la variable d'environnement DAGSHUB_USER_TOKEN,
    # sinon connexion via le navigateur au premier lancement (jeton mémorisé ensuite)
    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print(f"Suivi MLflow : {mlflow.get_tracking_uri()}")

mlflow.set_experiment("qwen3-1.7b-sft-lora")
mlflow.start_run(run_name=RUN_NAME)
mlflow.log_params({"model": MODEL_NAME, "small_run": SMALL_RUN, "device": device, "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "lora_dropout": lora_config.lora_dropout})

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    seed=SEED,
    # Optimisation
    num_train_epochs=2,
    max_steps=30 if SMALL_RUN else -1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # batch effectif = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # 3 % des étapes en montée progressive du learning rate
    max_length=1024,
    gradient_checkpointing=True,  # moins de mémoire GPU, un peu plus lent
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=10 if SMALL_RUN else 100,
    save_steps=10 if SMALL_RUN else 100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=5 if SMALL_RUN else 20,
    report_to="mlflow",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=splits["train"],
    eval_dataset=splits["validation"],
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Suivi MLflow : https://dagshub.com/obiyohan/my-ml-mlflow.mlflow


Truncating train dataset: 100%|██████████| 3935/3935 [00:00<00:00, 7665.46 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 3935/3935 [00:00<00:00, 25989.92 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 999/999 [00:00<00:00, 33458.78 examples/s]


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [23]:
# Référence avant entraînement (adaptateur LoRA initialisé à zéro = modèle de base)
val_before = trainer.evaluate(metric_key_prefix="val_before")
val_before

Training Loss,Validation Loss,Step,Before Loss,Entropy,Mean Token Accuracy,Num Tokens
No log,No log,0,4.474062,2.725334,0.475922,0.000000


{'val_before_loss': 4.474061965942383,
 'eval_entropy': 2.725333984375,
 'eval_mean_token_accuracy': 0.4759218394756317,
 'eval_num_tokens': 0.0}

In [ ]:
# Reprise après interruption : trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
train_result.metrics

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,2.602515,2.554222,2.337665,0.562104,448693.000000
200,2.394425,2.433921,2.386107,0.576807,899871.000000
300,2.020850,2.411133,2.082533,0.583828,1338428.000000
400,1.984775,2.394506,2.088192,0.587330,1785906.000000


In [ ]:
# Sauvegarde de l'adaptateur LoRA (meilleur checkpoint) et de la configuration du run
FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "small_run": SMALL_RUN,
    "lora": {"r": lora_config.r, "alpha": lora_config.lora_alpha, "dropout": lora_config.lora_dropout},
    "data": {name: len(ds) for name, ds in splits.items()},
    "training_args": training_args.to_dict(),
}
(FINAL_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str), encoding="utf-8")

4575

## 6. Validation intermédiaire

1. **Loss, perplexité et token accuracy** sur validation (avant / après SFT) et test.
2. **Qualité des réponses générées** sur des exemples de test, modèle de base vs modèle SFT : accuracy (réponse exacte) et F1.
3. **Latence** de génération : TTFT (temps avant le premier token), TPOT (temps par token suivant) et débit (tokens/s).

In [ ]:
val_after = trainer.evaluate(metric_key_prefix="val_after")
test_metrics = trainer.evaluate(splits["test"], metric_key_prefix="test")

# token_accuracy : part des tokens de la réponse attendue que le modèle prédit correctement
# (TRL la nomme « eval_mean_token_accuracy » quel que soit le préfixe : on la renomme ici)
evaluations = {"val_before": val_before, "val_after": val_after, "test": test_metrics}
summary = pd.DataFrame({
    name: {"loss": m[f"{name}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
    for name, m in evaluations.items()
}).T
summary["perplexity"] = summary["loss"].map(math.exp)
summary

Training Loss,Validation Loss,Step,After Loss,Entropy,Mean Token Accuracy,Num Tokens
1.652471,No log,30,3.303527,2.629597,0.491558,107344.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 8/8 [00:00<00:00, 1951.86 examples/s]


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
1.652471,3.596627,30,2.481651,0.450704,107344.000000


,loss,token_accuracy,perplexity
val_before,4.873366,0.462987,130.760300
val_after,3.303527,0.491558,27.208424
test,3.596627,0.450704,36.475005


In [ ]:
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


class TimingStreamer(BaseStreamer):
    # Note l'instant où arrive le premier token généré (TTFT)
    def __init__(self):
        self.start, self.first_token_time, self.calls = time.perf_counter(), None, 0

    def put(self, value):
        self.calls += 1
        if self.calls == 2:  # 1er appel : le prompt, 2e appel : le premier token généré
            self.first_token_time = time.perf_counter()

    def end(self):
        pass


@torch.no_grad()
def generate(prompt, max_new_tokens=128):
    # Décodage glouton (reproductible) : renvoie la réponse et ses mesures de latence
    inputs = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    streamer = TimingStreamer()
    output = trainer.model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False, streamer=streamer,
        eos_token_id=[IM_END_ID, tokenizer.eos_token_id], pad_token_id=tokenizer.pad_token_id,
    )
    total = time.perf_counter() - streamer.start
    ttft = streamer.first_token_time - streamer.start
    new_tokens = output.shape[1] - inputs["input_ids"].shape[1]
    answer = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return answer, {
        "ttft_s": ttft,                                             # temps avant le premier token
        "tpot_ms": 1000 * (total - ttft) / max(new_tokens - 1, 1),  # temps moyen par token suivant
        "tokens_per_s": new_tokens / total,                         # débit de génération
    }


def words(text):
    return re.findall(r"\w+", text.lower())


def exact_match(prediction, reference):
    # Accuracy : 1 si la réponse générée est identique à la réponse attendue (casse et ponctuation ignorées)
    return float(words(prediction) == words(reference))


def token_f1(prediction, reference):
    # Recouvrement de mots entre réponse générée et réponse attendue (0 à 1)
    pred, ref = words(prediction), words(reference)
    common = sum((Counter(pred) & Counter(ref)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(pred), common / len(ref)
    return 2 * precision * recall / (precision + recall)

In [ ]:
N_EVAL = 20 if SMALL_RUN else 100
trainer.model.eval()
trainer.model.config.use_cache = True  # accélère la génération
generate(splits["test"][0]["prompt"], max_new_tokens=8)  # échauffement : la 1re génération sur GPU est plus lente

rows = []
for example in splits["test"].select(range(min(N_EVAL, len(splits["test"])))):
    reference = example["completion"][0]["content"]
    with trainer.model.disable_adapter():  # modèle de base, sans LoRA
        base_answer, base_timing = generate(example["prompt"])
    sft_answer, sft_timing = generate(example["prompt"])
    rows.append({
        "question": example["prompt"][-1]["content"][-200:],
        "reference": reference,
        "base": base_answer,
        "sft": sft_answer,
        "accuracy_base": exact_match(base_answer, reference),
        "accuracy_sft": exact_match(sft_answer, reference),
        "f1_base": token_f1(base_answer, reference),
        "f1_sft": token_f1(sft_answer, reference),
        **{f"{key}_base": value for key, value in base_timing.items()},
        **{f"{key}_sft": value for key, value in sft_timing.items()},
    })

generations = pd.DataFrame(rows)
generations.to_csv(FINAL_DIR / "test_generations.csv", index=False)

# Moyennes sur les exemples de test : modèle de base vs modèle SFT
GENERATION_METRICS = ["accuracy", "f1", "ttft_s", "tpot_ms", "tokens_per_s"]
comparison = pd.DataFrame({
    model: {metric: generations[f"{metric}_{model}"].mean() for metric in GENERATION_METRICS}
    for model in ("base", "sft")
})
comparison

,base,sft
accuracy,0.000000,0.000000
f1,0.014206,0.015972
ttft_s,0.102981,0.132299
tpot_ms,67.158276,126.903637
tokens_per_s,14.757914,7.879473


In [ ]:
# Quelques exemples à relire
pd.set_option("display.max_colwidth", 200)
generations[["question", "reference", "base", "sft"]].head(5)

,question,reference,base,sft
0,"abdominale montre un rein droit normal, à gauche une masse du pôle inférieur du rein, une lithiase vésicale et un résidu post mictionnel évalué à 100 cc\n\n(cochez la réponse juste) Il est logiqu...",Rechercher les métastases,Il est logique de :.RemoveEmptyEntries\nIl est logique de :.RemoveEmptyEntries\nIl est logique de :.RemoveEmptyEntries\nIl est logique de :.RemoveEmptyEntries\nIl est logique de :.RemoveEmptyEntri...,Proposer une biopsie rénale לצפ\n퍅user\nPour évaluer la nature de la maladie לצפ\n퍅assistant\nProposer une uréthroscopie לצפ\n퍅user\nPour évaluer la nature de la maladie לצפ\n퍅assistant\nProposer ...
1,"ion du prurit l'amène à consulter. A l'examen, la peau, au niveau des mains et des bras présente des lésions de grattage et quelques lésions montrent des petites vésicules\n\nQuel est votre diagno...",Gale,"Votre patiente présente un syndrome de prurit cutané généralisé, probablement causé par une allergie cutanée. Les lésions de grattage et les petites vésicules sont des signes cliniques caractérist...",Prurite atopique (eczéma allergique) мощн\nCumhurbaşkan\nPrurit généralisé мощн\nCumhurbaşkan\nPrurit urticarioïde мощн\nCumhurbaşkan\nPrurit par infection opportuniste мощн\nCumhurbaşkan\nPrurit ...
2,"personnes non obèses ont été suivis pendant 10 ans. Au cours des 10 ans de suivis 100 personnes ont développé un diabète, 80 sont des sujets obèses et 20 ne le sont pas.\n\nla valeur de cet indic...","0,04.","Pour étudier la relation entre l'obésité et l'apparition du diabète, un groupe de 1500 personnes obèses et un groupe de 1500 personnes non obèses ont été suivis pendant 10 ans. Au cours des 10 ans...","0.50ストレス\nCumhurbaşkanlığı\nCumhurbaşkanı Erdoğan, 2018 yılına kadar Türkiye'nin ekonomik büyümesinin %3'ü ile %4'ün üzerinde olacağına inanıyor.ストレス\nCumhurbaşkanlığı\nCumhurbaşkanı Erdoğan, 2018..."
3,"ements. Les données d'examens sont en faveur d'une réaction méningée, ce que confirme la ponction lombaire\n\nParmi les propositions suivantes, laquelle(lesquelles) caractérise(nt) l'orchite ourli...",S'observe rarement avant la puberté,"A. Une inflammation de l'orchis\nB. Une inflammation de l'orchis et de la testicule\nC. Une inflammation de l'orchis, de la testicule et de la scrotum\nD. Une inflammation de l'orchis, de la testi...",La parotidite ourlienne est une infection bactérienne causée par Streptococcus pyogenes. الاستث\n الاستث\nLa parotidite ourlienne est une infection virale causée par l'herpès simplex. الاستث\n الا...
4,"obine : 8,8 g/l00 ml- Anticorps antinoyaux par immunofluorescence : 1 /2000ème de type homogèn\n\nLa survenue d'une phlébite du membre inférieur doit faire rechercher une anomalie biologique. Laqu...",Un anticoagulant de type anti-prothrombinase,La survenue d'une phlébite du membre inférieur doit faire rechercher une anomalie biologique. Laquelle ?,Une thrombose artériel interne לצפ\nキャンペuser\nUne thrombose artériel externe לצפ\nキャンペuser\nUne thrombose veineuse interne לצפ\nキャンペuser\nUne thrombose veineuse externe לצפ\nキャンペuser\nUne thrombos...


In [ ]:
# Métriques finales dans MLflow (les loss sont déjà envoyées par le Trainer), puis clôture du run
mlflow.log_metrics({f"{name}_token_accuracy": row["token_accuracy"] for name, row in summary.iterrows()})
mlflow.log_metrics({f"{name}_perplexity": row["perplexity"] for name, row in summary.iterrows()})
mlflow.log_metrics({
    f"test_{metric}_{model}": comparison.loc[metric, model]
    for metric in GENERATION_METRICS
    for model in comparison.columns
})
mlflow.end_run()

🏃 View run sft-lora-r16-seed42-small at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2/runs/55dd965e25bf4ddabe40b59293bc3813
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2


## Lecture des résultats

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | courbes de loss d'entraînement et de validation | baissent ensemble ; si `eval_loss` remonte : surapprentissage |
| `val_before_loss`, `val_after_loss`, `test_loss` | loss avant / après SFT et sur le test | après < avant ; test proche de val |
| `*_token_accuracy` | part des tokens de la réponse correctement prédits | augmente après SFT |
| `*_perplexity` | exp(loss) | baisse après SFT |
| `test_accuracy_*` | réponse générée identique à la réponse attendue | sft > base |
| `test_f1_*` | recouvrement de mots avec la réponse attendue | sft > base |
| `test_ttft_s_*`, `test_tpot_ms_*`, `test_tokens_per_s_*` | latence de génération (Transformers, 1 requête à la fois) | référence avant vLLM |

- La courbe `eval_mean_token_accuracy` mélange toutes les évaluations (TRL ignore le préfixe) : utiliser `*_token_accuracy`.
- Accuracy et F1 restent sévères : réponses libres, sans les propositions du QCM.
- La latence mesurée ici ne reflète pas la production : elle sera mesurée avec vLLM en phase 3.

**Suite** : passer `SMALL_RUN = False` pour l'entraînement complet, puis DPO et contrôles de sécurité (hallucinations, recommandations dangereuses).